# 03 — Transaction Preprocessing & Modeling Readiness

Prepare order baskets for Phase 2 without executing Apriori/FP-Growth. Explain feature types, scaling decision, and record-level data quality. The included source tables are a real 30,000-order sample, with complete baskets.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# Original Instacart, 30,000 reproducibly selected prior orders.
DATA_MODE = 'real'
print('DATA MODE:', DATA_MODE.upper(), '|', 'REAL INSTACART SAMPLE' if DATA_MODE == 'demo' else 'ORIGINAL INSTACART DATA')

DATA MODE: REAL | ORIGINAL INSTACART DATA


## Ensure reproducible processed tables
Each basket stores a unique product-ID list, delimited by `|`, derived from complete selected prior orders. `user_id` is only a customer reference and not a numerical quantity.

In [2]:
import pandas as pd, json
from IPython.display import display
processed=ROOT/'data/processed'
# To regenerate: python -m scripts.run_phase1 --max-orders 30000
baskets=pd.read_csv(processed/'baskets.csv')
report=json.loads((processed/'cleaning_report.json').read_text())
print('Unique baskets:',len(baskets),'Empty baskets:',int((baskets.basket_size==0).sum()))
display(baskets.head(10))
out={name: pd.read_csv(processed/f'{name}.csv',nrows=10) for name in ['orders','product_catalog','order_products_clean','baskets']}

Unique baskets: 30000 Empty baskets: 0


,order_id,user_id,basket_size,product_ids,product_names,order_dow,order_hour_of_day
0,136,184355,4,11005|18465|29926|47766,Organic Avocado | Organic Cream Cheese Bar | O...,6,10
1,376,121858,7,890|2716|8277|16462|35851|45210|47029,Apple Honeycrisp Organic | Organic Diced Tomat...,5,16
2,522,35250,20,1862|2344|10251|13292|17706|24184|24852|25340|...,Africa Single Origin 1 Coffee | Banana | Cryst...,1,15
3,526,17900,18,1752|1831|1940|9387|12191|12302|22489|23085|27...,"Beef Franks | Country Stand Juice, Medium Pulp...",0,23
4,552,18926,4,4091|5268|33452|37011,Artichokes | Bagel Bites Three Cheese Mini Bag...,4,13
5,776,149408,7,1216|3740|18465|18883|26452|28934|31526,Dairy Free Gluten Free Cheeze Lover's Pizza | ...,0,20
6,873,189247,12,2380|4605|7021|8518|20114|21070|28535|31717|35...,Asparagus | Cayenne Pepper | Cucumber & Garlic...,0,10
7,911,172302,11,19457|19822|20084|25653|26882|30459|32931|3298...,Egg Challah Rolls | Four Fruits 100% Fruit Spr...,2,12
8,1135,201623,22,3957|5134|10017|11737|17600|19816|22035|24852|...,100% Raw Coconut Water | Banana | Bartlett Pea...,2,7
9,1348,44705,10,5785|6182|8277|13176|15182|22115|36070|39921|4...,Apple Honeycrisp Organic | Bag of Organic Bana...,6,11


## Preprocessing justification
- Data for association mining is **categorical transactional presence**; product IDs are labels, not continuous numeric features.
- Do **not** apply StandardScaler or MinMaxScaler to IDs, and do **not** impute absent items as missing values.
- Phase 2 may one-hot encode product presence; that transforms category membership, not normalized magnitudes.
- Reordered status can be used to describe history, but does not create a supervised target for association-rule mining.
- For genuine evaluation later, hold out whole transactions and avoid leakage between training rules and evaluation baskets.

In [3]:
print('Product ID examples (categorical):', baskets.product_ids.head(3).tolist())
print('Scaling: NOT APPLICABLE — product ID baskets become binary presence indicators in Phase 2.')
print('Rows exported:', {k:len(v) for k,v in out.items()})

Product ID examples (categorical): ['11005|18465|29926|47766', '890|2716|8277|16462|35851|45210|47029', '1862|2344|10251|13292|17706|24184|24852|25340|25837|26209|30391|30489|31130|31444|32611|34126|35939|41950|45200|46676']
Scaling: NOT APPLICABLE — product ID baskets become binary presence indicators in Phase 2.
Rows exported: {'orders': 10, 'product_catalog': 10, 'order_products_clean': 10, 'baskets': 10}
